In [ ]:
# Building Trustworthy Insights with BERT — single Colab cell

# 0) Install dependencies
!pip -q install -U datasets transformers accelerate evaluate scikit-learn matplotlib

# 1) Imports
import os, math, numpy as np, matplotlib.pyplot as plt
import torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          AutoModel, Trainer, TrainingArguments, DataCollatorWithPadding)
import evaluate

# 2) Data loading & inspection (tweet_eval: sentiment)
ds = load_dataset("tweet_eval", "sentiment")
label_names = ds["train"].features["label"].names  # ['negative','neutral','positive']
label2id = {name:i for i,name in enumerate(label_names)}
id2label = {i:name for name,i in label2id.items()}
print("Splits:", {k: len(v) for k,v in ds.items()})
for split in ds:
    counts = np.bincount(ds[split]["label"], minlength=len(label_names))
    print(f"{split} distribution:", dict(zip(label_names, counts.tolist())))

# save two example tweets per label for later attention viz
examples_by_label = {i: [] for i in range(len(label_names))}
for ex in ds["validation"]:
    if len(examples_by_label[ex["label"]]) < 2:
        examples_by_label[ex["label"]].append(ex["text"])
    if all(len(v)==2 for v in examples_by_label.values()):
        break
print("Saved examples:", {id2label[k]: v for k,v in examples_by_label.items()})

# 3) Tokenization pipeline
model_ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
max_len = 128

def preprocess(batch):
    return tokenizer(batch["text"], truncation=True, padding=False, max_length=max_len)

encoded = ds.map(preprocess, batched=True, remove_columns=["text"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Set torch format
encoded.set_format(type="torch", columns=["input_ids","attention_mask","label"])

# 4) Fine-tuning setup (DistilBERT)
num_labels = len(label_names)
model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt, num_labels=num_labels, id2label=id2label, label2id=label2id
)

accuracy = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy.compute(predictions=preds, references=labels)["accuracy"]
    f1_macro = f1_metric.compute(predictions=preds, references=labels, average="macro")["f1"]
    return {"accuracy": acc, "f1": f1_macro}

out_dir = "distilbert-sentiment"
args = TrainingArguments(
    output_dir=out_dir,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    num_train_epochs=3,
    learning_rate=5e-5,
    weight_decay=0.01,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    logging_steps=100,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=encoded["train"],
    eval_dataset=encoded["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

train_result = trainer.train()
print("Best checkpoint:", trainer.state.best_model_checkpoint)

# 5) Evaluation & calibration
val_metrics = trainer.evaluate(encoded["validation"])
test_metrics = trainer.evaluate(encoded["test"])
print("Validation:", val_metrics)
print("Test:", test_metrics)

# collect softmax confidences on test set
with torch.no_grad():
    confs = []
    preds_all = []
    for batch in torch.utils.data.DataLoader(encoded["test"], batch_size=64):
        logits = trainer.model(input_ids=batch["input_ids"].to(trainer.model.device),
                               attention_mask=batch["attention_mask"].to(trainer.model.device)).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()
        pred_conf = probs.max(axis=-1)  # confidence of predicted class
        confs.append(pred_conf)
        preds_all.append(probs.argmax(axis=-1))
    confs = np.concatenate(confs)
    preds_all = np.concatenate(preds_all)

plt.figure(figsize=(6,4))
plt.hist(confs, bins=[i/10 for i in range(11)], edgecolor='black')
plt.title("Predicted-class confidence histogram (test)")
plt.xlabel("Confidence")
plt.ylabel("Count")
plt.show()

# 6) Attention inspection (DistilBERT encoder, last layer [CLS]→tokens)
encoder = AutoModel.from_pretrained(model_ckpt, output_attentions=True)
encoder.eval()

def cls_attention_scores(text):
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len)
    with torch.no_grad():
        out = encoder(**enc)
    # last layer attentions: tuple(len=layers) of (batch, heads, seq, seq)
    attn_last = out.attentions[-1][0]                   # (heads, seq, seq)
    attn_avg_heads = attn_last.mean(dim=0)              # (seq, seq)
    # from CLS (position 0) to each token
    scores = attn_avg_heads[0].cpu().numpy()            # (seq,)
    tokens = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    return tokens, scores

example_text = examples_by_label[label2id["negative"]][0] if examples_by_label[label2id["negative"]] else "terrible service and rude staff"
tokens, scores = cls_attention_scores(example_text)
# normalize and show top tokens excluding special tokens
mask = [i for i,tok in enumerate(tokens) if tok not in ("[CLS]","[SEP]","[PAD]")]
top_idx = np.argsort(scores[mask])[::-1][:10]
top_tokens = [tokens[mask[i]] for i in top_idx]
top_scores = [scores[mask[i]] for i in top_idx]
plt.figure(figsize=(8,3))
plt.bar(range(len(top_tokens)), top_scores)
plt.xticks(range(len(top_tokens)), top_tokens, rotation=45, ha='right')
plt.title("[CLS] attention to tokens (last layer)")
plt.tight_layout()
plt.show()
print("Example text:", example_text)
print("Top tokens by [CLS] attention:", top_tokens)

# 7) Inference helper with highlighted tokens (using attention as proxy)
label_map = id2label

def analyze_text(text, k=5):
    # classifier prediction
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_len, padding="max_length")
    with torch.no_grad():
        logits = trainer.model(**{k: v.to(trainer.model.device) for k,v in enc.items()}).logits
        probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]
    pred_id = int(probs.argmax())
    label = label_map[pred_id]
    confidence = float(probs[pred_id])

    # attention-based highlights from encoder
    with torch.no_grad():
        out = encoder(input_ids=enc["input_ids"], attention_mask=enc["attention_mask"])
    attn_last = out.attentions[-1][0]         # (heads, seq, seq)
    attn_avg = attn_last.mean(dim=0)[0].cpu().numpy()  # (seq,)
    toks = tokenizer.convert_ids_to_tokens(enc["input_ids"][0])
    keep = [(i,t) for i,t in enumerate(toks) if t not in ("[CLS]","[SEP]","[PAD]")]
    idxs = [i for i,_ in keep]
    toks_clean = [t for _,t in keep]
    topk_idx = np.argsort(attn_avg[idxs])[::-1][:k]
    highlighted_tokens = [toks_clean[i] for i in topk_idx]

    return {"label": label, "confidence": confidence, "highlighted_tokens": highlighted_tokens}

demo = analyze_text("The app kept crashing, support never replied, awful experience.")
print(demo)

# 8) Save artifacts
save_dir = "distilbert-sentiment-best"
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print("Saved to:", os.listdir(save_dir) if os.path.exists(save_dir) else "not saved")
